# Building an MCP client
Companion notebook for the **Building an MCP client** section of Chapter 5.

Connects to the search server from notebook 03 using two transports: **STDIO** (the client launches the server as a local subprocess) and **SSE** (the client connects to an already-running server over HTTP).

> **Run notebook 03 first** so `server.py` exists in this folder -- the STDIO client launches it directly.

Jupyter already runs an asyncio event loop, so cells below use top-level `await` directly instead of the book's `asyncio.run(main())` (which cannot be called from inside a loop that's already running).

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.sse import sse_client
from langchain_mcp_adapters.tools import load_mcp_tools

load_dotenv()

False

## Connecting via STDIO
The client starts `server.py` itself as a subprocess and talks to it over stdin/stdout.

In [2]:
async def run_stdio():
    print("=== Connecting via STDIO ===")

    # Pass environment variables so the server subprocess can read TAVILY_API_KEY / OPENAI_API_KEY
    env = os.environ.copy()
    server_path = Path.cwd() / "server.py"

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(server_path), "stdio"],
        env=env,
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await load_mcp_tools(session)
            print("Available Tools:")
            for t in tools:
                print(f" - {t.name}: {t.description}")

            wiki_tool = next((t for t in tools if t.name == "search_wikipedia"), None)
            if wiki_tool:
                print("\nInvoking 'search_wikipedia' tool...")
                result = await wiki_tool.ainvoke({"query": "Model Context Protocol"})
                print(f"Result snippet:\n{str(result)[:400]}...\n")

            web_tool = next((t for t in tools if t.name == "search_web"), None)
            if web_tool:
                print("Invoking 'search_web' tool (via Tavily)...")
                try:
                    result = await web_tool.ainvoke({"query": "Recent breakthroughs in AI agents"})
                    print(f"Result snippet:\n{str(result)[:600]}...\n")
                except Exception as e:
                    print(f"Search failed. Did you export TAVILY_API_KEY? Error: {e}")

In [3]:
await run_stdio()

=== Connecting via STDIO ===


Available Tools:
 - search_wikipedia: Search Wikipedia for information about people, places, facts, and history.
 - search_web: Search the web for current events, news, or general queries using Tavily.

Invoking 'search_wikipedia' tool...


Result snippet:
[{'type': 'text', 'text': "Page: Model Context Protocol\nSummary: The Model Context Protocol (MCP) is an open standard and open-source framework introduced by Anthropic in November 2024 to standardize the way artificial intelligence (AI) systems like large language models (LLMs) integrate and share data with external tools, systems, and data sources. MCP provides a standardized interface for readi...

Invoking 'search_web' tool (via Tavily)...
Result snippet:
[{'type': 'text', 'text': 'TAVILY_API_KEY not set - cannot search the web.', 'id': 'lc_d770d94d-765a-4dd2-b96b-315f041bc0fc'}]...



## Connecting via SSE
SSE connects to a server URL instead of launching a subprocess -- ideal for a remote or shared server. This needs `server.py` running separately first:
```bash
python server.py sse
```
Run that in a separate terminal before running the cell below. If nothing is listening on `localhost:8000`, the `try/except` reports the connection failure instead of crashing the notebook -- exactly as the book's own client code handles it.

In [4]:
async def run_sse():
    print("=== Connecting via SSE ===")
    print("NOTE: start the server first in a separate terminal: `python server.py sse`")
    url = "http://localhost:8000/sse"

    try:
        async with sse_client(url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                tools = await load_mcp_tools(session)
                print("Available Tools from SSE:")
                for t in tools:
                    print(f" - {t.name}")

                wiki_tool = next((t for t in tools if t.name == "search_wikipedia"), None)
                if wiki_tool:
                    print("\nInvoking 'search_wikipedia' tool...")
                    result = await wiki_tool.ainvoke({"query": "Model Context Protocol"})
                    print(f"Result snippet:\n{str(result)[:250]}...")
    except Exception as e:
        print(f"Could not connect via SSE. Error: {e}")

In [5]:
await run_sse()

=== Connecting via SSE ===
NOTE: start the server first in a separate terminal: `python server.py sse`


Could not connect via SSE. Error: unhandled errors in a TaskGroup (1 sub-exception)
